## Quality control per cluster, and add platform adjusted ic_id_sample

Library

In [1]:
# Path-related libraries
import os
from pyhere import here  # For reproducible relative paths
import sys # system specific parameters
from pathlib import Path # File system paths

# AnnData and single-cell analysis libraries
import scanpy as sc       # For preprocessing and visualization of single-cell data
import anndata as ad      # For handling AnnData objects

# Numerical operations
import numpy as np        # For numerical computations and array manipulations
import pandas as pd

# Custom modules and functions
#sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
#import my_anndata as ma                    # Custom AnnData utilities

Parameters

In [2]:
# Path to save Anndata object
# Paths
base_dir = str(here('data/quality_control_per_cluster/first_pass/'))
files_dir = os.path.join(base_dir, 'files') 
anndata_dir = str(here('data/anndata/'))

Load data

In [3]:
# A data object
adata = ad.read_h5ad(os.path.join(anndata_dir, 'AB_combined.h5ad'))

# Cells that failed QC
cells_keep = pd.read_csv(os.path.join(files_dir, 'barcodes_pass_qc.csv'))

Keep only cells that passed per cluster quality control

In [28]:
# subset cells but keep all rows
mask = adata.obs.index.isin(cells_keep['barcode'])

# copy to avoid view warning
adata_sub = adata[mask, :].copy()

Number of cells removed

In [29]:
adata.shape[0] - adata_sub.shape[0]

86339

Rename columns so that names are more meaningfull (as is done i AC_combined.h5ad) and add platform adjusted ID 

In [30]:
# Extract old meta data
old_meta = adata_sub.obs.copy()

# Rename
adata_sub.obs.rename(columns={'ic_id_study': 'ic_id_dataset'}, inplace=True) # Current id reflects dataset not study
adata_sub.obs.rename(columns={'ic_id_study_overall': 'ic_id_study'}, inplace=True) # Now overall study can be called study
adata_sub.obs.rename(columns={'ic_id_donor': 'ic_id_dataset_donor'}, inplace=True) # Now id reflects donor per dataset

# Check renaming was done correctly
assert (old_meta['ic_id_study'] == adata_sub.obs['ic_id_dataset']).all()
assert (old_meta['ic_id_study_overall'] == adata_sub.obs['ic_id_study']).all()
assert (old_meta['ic_id_donor'] == adata_sub.obs['ic_id_dataset_donor']).all()

# Add platform adjusted id 
adata_sub.obs['ic_id_platform_adjusted_sample'] = np.where(
    adata_sub.obs['platform'].isin(['droplet', 'plate_barcode']),
    adata_sub.obs['ic_id_sample'],
    np.where(adata_sub.obs['platform'].isin(['plate']),
             adata_sub.obs['ic_id_dataset_donor'],
             None)  # fallback if platform is something else
)

# Check that 'ic_id_platform_adjusted_sample' column was added
assert 'ic_id_platform_adjusted_sample' in adata_sub.obs.columns

In [44]:
print(f'Number of studies: {adata_sub.obs['ic_id_study'].value_counts().count()}')
print(f'Number of datasets: {adata_sub.obs['ic_id_dataset'].value_counts().count()}')
print(f'Number of unique donors: {adata_sub.obs['ic_id_donor_overall'].value_counts().count()}')
print(f'Number of samples: {adata_sub.obs['ic_id_platform_adjusted_sample'].value_counts().count()}')

Number of studies: 22
Number of datasets: 25
Number of unique donors: 270
Number of samples: 333


Save anndata object

In [45]:
adata_sub.write(os.path.join(anndata_dir, 'AE_combined.h5ad'))

Save new meta data